In [12]:
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv
from tqdm import tqdm
import json
import os

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [13]:
df = pd.read_csv("data/clean_jobs.csv")

In [14]:
df["experience_level"] = df["experience_level"].astype(str)
df["education"] = df["education"].astype(str)

### Using AI to fill in Education and Experiemce based on Job title and skills required

In [17]:
for idx, row in tqdm(df.iterrows(), total=len(df)):
    
    title = row["title"]
    skills = row["skills"]

    prompt = f"""Given this job posting:
Title: {title}
Skills: {skills}

Return ONLY a JSON object with exactly these fields:
{{
    "experience_level": "one of: entry-level, mid-level, senior, lead",
    "education": "one of: Bachelors in <field>, Masters in <field>, PhD in <field>"
}}
No explanation, no markdown, just the JSON."""

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens = 50,
            messages=[
                {
                    "role": "user", 
                    "content": prompt
                    }
            ]
        )

    
        result = response.choices[0].message.content
        data = json.loads(result)

        df.at[idx, "experience_level"] = data["experience_level"]
        df.at[idx, "education"] = data["education"]
    except Exception as e:
        print(f"Row {idx} failed: {e}")

100%|██████████| 4284/4284 [53:07<00:00,  1.34it/s]  


In [19]:
df.head()

,title,company,location,work_mode,job_type,skills,salary_avg,date_posted,experience_level,education,source,job_url,salary_min,salary_max
0,Data Scientist,"Comscore, Inc.","Pune, Maharashtra, India",onsite,fulltime,"['Linux', 'Python', 'R', 'SQL', 'Spark']",NaN,2026-03-09 00:00:00+00:00,mid-level,Masters in Data Science,datasciencejobs,https://datasciencejobs.com/jobs/data-scientis...,NaN,NaN
1,"Senior Data Scientist, Agentic AI (Insurance U...",Guardian Life,"New York, New York, United States",onsite,fulltime,"['Git', 'PyTorch', 'Python', 'TensorFlow']",157222.5,2026-03-09 00:00:00+00:00,senior,Masters in Data Science,datasciencejobs,https://datasciencejobs.com/jobs/senior-data-s...,118980.0,195465.0
2,Financial Crimes - Senior Data Scientist,KeyBank,"Cleveland, Ohio, United States",onsite,fulltime,"['Python', 'SAS']",134500.0,2026-03-09 00:00:00+00:00,senior,Masters in Data Science,datasciencejobs,https://datasciencejobs.com/jobs/financial-cri...,94000.0,175000.0
3,"Research Scientist – Computer Graphics, Vision...",Eyeline,"Los Angeles, California, United States",onsite,fulltime,['Python'],210000.0,2026-03-09 00:00:00+00:00,mid-level,Masters in Computer Science,datasciencejobs,https://datasciencejobs.com/jobs/research-scie...,100000.0,320000.0
4,Emerging Talent – Computer Vision Engineering ...,Baker Hughes,"Florence, Tuscany, Italy",onsite,fulltime,['Python'],NaN,2026-03-09 00:00:00+00:00,entry-level,Bachelors in Computer Science,datasciencejobs,https://datasciencejobs.com/jobs/emerging-tale...,NaN,NaN


In [ ]:
df.to_csv("data/clean/filled_jobs.csv", index=False)